<a href="https://colab.research.google.com/github/woochanj816/handson_ch4/blob/main/Softmax_Regression_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Softmax Regression from Scratch

Exercise 12: 조기종료를 사용한 배치 경사 하강법으로 소프트맥스 회기 구현하기(사이킷런x)

Main goals:

- Implement **Softmax** function
- Implement Cross-Entropy **Loss**
- Implement Batch **Gradient Descent**
- Add L2 Regularization
- Use **Early Stopping**
- Visualize Learning Curve and Decision Boundary

라이브러리 불러오기

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# sklearn 사용 x

CSV 파일에서 Iris 데이터를 읽고, 입력값 X와 정답 라벨 y로 나누는 부분

In [ ]:
def load_iris_from_csv(path="iris.csv"): # csv에서 iris 데이터를 불러오는 함수
    df = pd.read_csv(path)

    X = df[["petal_length", "petal_width"]].values
        # 길이/너비 두 특성만 사용 (2차원 평면으로 시각화하여 나타내기 위해)
    label_map = {
        "Iris-setosa": 0,
        "Iris-versicolor": 1,
        "Iris-virginica": 2,
        "setosa": 0,
        "versicolor": 1,
        "virginica": 2,
    }
        # 문자열을 0, 1, 2의 숫자 라벨로 변환

    y = df["species"].map(label_map).values

    return X, y

검증/테스트 데이터 나누기 + one hot Encoding

In [ ]:
def train_val_test_split(X, y, train_ratio=0.6, val_ratio=0.2, seed=44):
        # 60% train, 20% validation, seed 고정
    np.random.seed(seed)

    indices = np.random.permutation(len(X))
    X = X[indices]
    y = y[indices]

    train_end = int(len(X) * train_ratio)
    val_end = int(len(X) * (train_ratio + val_ratio))

    X_train = X[:train_end]
    y_train = y[:train_end]

    X_val = X[train_end:val_end]
    y_val = y[train_end:val_end]

    X_test = X[val_end:]
    y_test = y[val_end:]

    return X_train, X_val, X_test, y_train, y_val, y_test


def add_bias(X):
    return np.c_[np.ones((X.shape[0], 1)), X]  # 입력 행렬에 1 추가 -> 열 늘어남

def one_hot(y, n_classes):
    return np.eye(n_classes)[y]
    # 정답 라벨을 one-hot vector로 바꿈

In [ ]:
X, y = load_iris_from_csv("iris.csv")

n_classes = len(np.unique(y))
Y = one_hot(y, n_classes)

print("y shape:", y.shape)
print("Y one-hot shape:", Y.shape)

print("unique y:", np.unique(y))
print("first 10 y:", y[:10])
print("first 10 one-hot:")
print(Y[:10])

FileNotFoundError: [Errno 2] No such file or directory: 'iris.csv'

Softmax 함수

In [ ]:
def softmax(logits):
    """
    logits: shape = (m, n_classes)
    return: class probabilities
    """

    # numerical stability를 위해 각 행에서 max를 빼줌
    shifted_logits = logits - np.max(logits, axis=1, keepdims=True)

    exp_logits = np.exp(shifted_logits)

    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)


def cross_entropy_loss(X, Y, theta, alpha=0.0):
    """
    X: input matrix with bias, shape = (m, n_features)
    Y: one-hot labels, shape = (m, n_classes)
    theta: parameter matrix, shape = (n_features, n_classes)
    alpha: L2 regularization strength
    """

    m = X.shape[0]

    logits = X.dot(theta)
    probabilities = softmax(logits)

    epsilon = 1e-15

    data_loss = -np.mean(
        np.sum(Y * np.log(probabilities + epsilon), axis=1)
    )

    # bias term theta[0]은 regularization에서 제외
    l2_loss = 0.5 * alpha * np.sum(theta[1:] ** 2)

    return data_loss + l2_loss


def compute_gradient(X, Y, theta, alpha=0.0):
    m = X.shape[0]

    logits = X.dot(theta)
    probabilities = softmax(logits)

    gradient = (1 / m) * X.T.dot(probabilities - Y)

    # bias term 제외하고 L2 regularization 적용
    gradient[1:] += alpha * theta[1:]

    return gradient


def predict(X, theta):
    logits = X.dot(theta)
    probabilities = softmax(logits)

    return np.argmax(probabilities, axis=1)


def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

fd




# 1. Data

def load_iris_from_csv(path="iris.csv"):
    df = pd.read_csv(path)

    # 2D decision boundary를 그리기 위해 꽃잎 길이, 꽃잎 너비만 사용
    X = df[["petal_length", "petal_width"]].values

    label_map = {
        "Iris-setosa": 0,
        "Iris-versicolor": 1,
        "Iris-virginica": 2,
        "setosa": 0,
        "versicolor": 1,
        "virginica": 2,
    }

    y = df["species"].map(label_map).values

    return X, y


def train_val_test_split(X, y, train_ratio=0.6, val_ratio=0.2, seed=42):
    np.random.seed(seed)

    indices = np.random.permutation(len(X))
    X = X[indices]
    y = y[indices]

    train_end = int(len(X) * train_ratio)
    val_end = int(len(X) * (train_ratio + val_ratio))

    X_train = X[:train_end]
    y_train = y[:train_end]

    X_val = X[train_end:val_end]
    y_val = y[train_end:val_end]

    X_test = X[val_end:]
    y_test = y[val_end:]

    return X_train, X_val, X_test, y_train, y_val, y_test


def add_bias(X):
    return np.c_[np.ones((X.shape[0], 1)), X]


def one_hot(y, n_classes):
    return np.eye(n_classes)[y]


# ============================================================
# 2. Softmax Regression
# ============================================================

def softmax(logits):
    """
    logits: shape = (m, n_classes)
    return: class probabilities
    """

    # numerical stability를 위해 각 행에서 max를 빼줌
    shifted_logits = logits - np.max(logits, axis=1, keepdims=True)

    exp_logits = np.exp(shifted_logits)

    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)


def cross_entropy_loss(X, Y, theta, alpha=0.0):
    """
    X: input matrix with bias, shape = (m, n_features)
    Y: one-hot labels, shape = (m, n_classes)
    theta: parameter matrix, shape = (n_features, n_classes)
    alpha: L2 regularization strength
    """

    m = X.shape[0]

    logits = X.dot(theta)
    probabilities = softmax(logits)

    epsilon = 1e-15

    data_loss = -np.mean(
        np.sum(Y * np.log(probabilities + epsilon), axis=1)
    )

    # bias term theta[0]은 regularization에서 제외
    l2_loss = 0.5 * alpha * np.sum(theta[1:] ** 2)

    return data_loss + l2_loss


def compute_gradient(X, Y, theta, alpha=0.0):
    m = X.shape[0]

    logits = X.dot(theta)
    probabilities = softmax(logits)

    gradient = (1 / m) * X.T.dot(probabilities - Y)

    # bias term 제외하고 L2 regularization 적용
    gradient[1:] += alpha * theta[1:]

    return gradient


def predict(X, theta):
    logits = X.dot(theta)
    probabilities = softmax(logits)

    return np.argmax(probabilities, axis=1)


def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)


# ============================================================
# 3. Batch Gradient Descent + Early Stopping
# ============================================================

def train_softmax_regression(
    X_train,
    Y_train,
    X_val,
    Y_val,
    learning_rate=0.1,
    alpha=0.01,
    n_epochs=5000,
    patience=100,
    seed=42
):
    np.random.seed(seed)

    n_features = X_train.shape[1]
    n_classes = Y_train.shape[1]

    theta = np.random.randn(n_features, n_classes) * 0.01

    best_theta = theta.copy()
    best_val_loss = float("inf")
    patience_count = 0

    train_losses = []
    val_losses = []

    for epoch in range(n_epochs):
        gradient = compute_gradient(X_train, Y_train, theta, alpha)
        theta = theta - learning_rate * gradient

        train_loss = cross_entropy_loss(X_train, Y_train, theta, alpha)
        val_loss = cross_entropy_loss(X_val, Y_val, theta, alpha)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_theta = theta.copy()
            patience_count = 0
        else:
            patience_count += 1

        if patience_count >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    return best_theta, train_losses, val_losses


# ============================================================
# 4. Visualization
# ============================================================

def plot_learning_curve(train_losses, val_losses):
    plt.figure(figsize=(8, 5))
    plt.plot(train_losses, label="Training Loss")
    plt.plot(val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Cross-Entropy Loss")
    plt.title("Learning Curve")
    plt.legend()
    plt.grid(True)
    plt.show()


def plot_decision_boundary(X, y, theta):
    x0_min, x0_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    x1_min, x1_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

    x0, x1 = np.meshgrid(
        np.linspace(x0_min, x0_max, 300),
        np.linspace(x1_min, x1_max, 300)
    )

    X_grid = np.c_[x0.ravel(), x1.ravel()]
    X_grid_bias = add_bias(X_grid)

    y_pred_grid = predict(X_grid_bias, theta)
    y_pred_grid = y_pred_grid.reshape(x0.shape)

    plt.figure(figsize=(8, 5))
    plt.contourf(x0, x1, y_pred_grid, alpha=0.3)
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors="k")

    plt.xlabel("Petal Length")
    plt.ylabel("Petal Width")
    plt.title("Softmax Regression Decision Boundary")
    plt.show()


# ============================================================
# 5. Main
# ============================================================

def main():
    X, y = load_iris_from_csv("iris.csv")

    X_train, X_val, X_test, y_train, y_val, y_test = train_val_test_split(X, y)

    X_train = add_bias(X_train)
    X_val = add_bias(X_val)
    X_test = add_bias(X_test)

    n_classes = len(np.unique(y))

    Y_train = one_hot(y_train, n_classes)
    Y_val = one_hot(y_val, n_classes)

    theta, train_losses, val_losses = train_softmax_regression(
        X_train,
        Y_train,
        X_val,
        Y_val,
        learning_rate=0.1,
        alpha=0.01,
        n_epochs=5000,
        patience=100
    )

    y_test_pred = predict(X_test, theta)
    test_acc = accuracy(y_test, y_test_pred)

    print(f"Test Accuracy: {test_acc:.4f}")

    plot_learning_curve(train_losses, val_losses)

    # 원래 X는 bias 없는 2차원 데이터가 필요함
    X_original, y_original = load_iris_from_csv("iris.csv")
    plot_decision_boundary(X_original, y_original, theta)


if __name__ == "__main__":
    main()